In [1]:
# ============================================================================
# HOMEWORK 4: CATBOOST + LIGHTGBM STACKING WORKFLOW
# ============================================================================
# Assignment map:
# 1. Build two meaningfully different tuned models: CatBoost and LightGBM.
# 2. Create and test new feature representations.
# 3. Evaluate which features are useful and why.
# 4. Stack CatBoost and LightGBM and compare the stack against each model.
#
# Primary validation metric: balanced_accuracy, because the target is imbalanced.
# ============================================================================

In [2]:
import warnings
warnings.filterwarnings("ignore")

import time

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
import lightgbm as lgb

from sklearn.base import clone
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

RANDOM_STATE = 42
TARGET_COL = "Irrigation_Need"
ID_COL = "id"
PRIMARY_METRIC = "balanced_accuracy"

In [3]:
# ============================================================================
# PART 1: LOAD DATA AND CREATE A SHARED VALIDATION SPLIT
# ============================================================================

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

train_clean = train.drop(columns=[ID_COL]).copy()
test_clean = test.drop(columns=[ID_COL]).copy()

y_all = train_clean[TARGET_COL].copy()
X_all_raw = train_clean.drop(columns=[TARGET_COL]).copy()

train_idx, valid_idx = train_test_split(
    np.arange(len(y_all)),
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_all,
    shuffle=True,
)

print("=" * 70)
print("DATA OVERVIEW")
print("=" * 70)
print(f"Train shape: {train_clean.shape}")
print(f"Test shape: {test_clean.shape}")
print("\nClass proportions:")
print(y_all.value_counts(normalize=True).round(4))
print(f"\nMissing values in train: {train_clean.isnull().sum().sum()}")
print(f"Missing values in test: {test_clean.isnull().sum().sum()}")
print(f"\nShared split: {len(train_idx):,} train rows | {len(valid_idx):,} validation rows")

DATA OVERVIEW
Train shape: (630000, 20)
Test shape: (270000, 19)

Class proportions:
Irrigation_Need
Low       0.5872
Medium    0.3795
High      0.0333
Name: proportion, dtype: float64

Missing values in train: 0
Missing values in test: 0

Shared split: 504,000 train rows | 126,000 validation rows


In [4]:
# ============================================================================
# PART 2: FEATURE ENGINEERING AND MODEL-SPECIFIC REPRESENTATIONS
# ============================================================================


def add_domain_features(df):
    """Create irrigation-specific features plus Homework3-style transformations."""
    out = df.copy()
    area = out["Field_Area_hectare"].replace(0, np.nan)

    # Homework4 irrigation/domain features.
    out["Water_Available"] = out["Rainfall_mm"] + out["Previous_Irrigation_mm"]
    out["Moisture_Deficit"] = out["Water_Available"] - out["Soil_Moisture"]
    out["Heat_Dryness"] = out["Temperature_C"] * (100 - out["Humidity"])
    out["Rainfall_per_Hectare"] = (out["Rainfall_mm"] / area).replace([np.inf, -np.inf], np.nan).fillna(0)
    out["Previous_Irrigation_per_Hectare"] = (
        out["Previous_Irrigation_mm"] / area
    ).replace([np.inf, -np.inf], np.nan).fillna(0)

    out["Crop_Stage"] = out["Crop_Type"].astype(str) + "__" + out["Crop_Growth_Stage"].astype(str)
    out["Soil_Crop"] = out["Soil_Type"].astype(str) + "__" + out["Crop_Type"].astype(str)

    # Homework3-style transformations for skew-heavy fields.
    out["log_Rainfall_mm"] = np.log1p(out["Rainfall_mm"].clip(lower=0))
    out["log_Previous_Irrigation_mm"] = np.log1p(out["Previous_Irrigation_mm"].clip(lower=0))
    out["temp_sq"] = out["Temperature_C"] ** 2

    # Homework3-style interactions.
    out["crop_soil_combo"] = out["Crop_Type"].astype(str) + "__" + out["Soil_Type"].astype(str)
    out["humidity_x_temp"] = out["Humidity"] * out["Temperature_C"]
    out["moisture_x_temp"] = out["Soil_Moisture"] * out["Temperature_C"]

    # Homework3-style grouped bins. qcut can create slightly different bin counts
    # when duplicate values exist, so convert to strings for categorical handling.
    out["field_area_bin"] = pd.qcut(
        out["Field_Area_hectare"], q=5, labels=False, duplicates="drop"
    ).astype(str)
    out["rainfall_bin"] = pd.qcut(
        out["Rainfall_mm"], q=5, labels=False, duplicates="drop"
    ).astype(str)

    # Homework3-style binary indicators.
    out["high_humidity"] = (out["Humidity"] >= out["Humidity"].median()).astype(int)
    out["hot_temp"] = (out["Temperature_C"] >= out["Temperature_C"].quantile(0.75)).astype(int)
    out["low_soil_moisture"] = (out["Soil_Moisture"] <= out["Soil_Moisture"].quantile(0.25)).astype(int)

    return out


def multiclass_target_encode(train_cat, test_cat, y, train_indices, valid_indices, n_splits=5):
    """Validation-safe multiclass target encoding.

    Training rows receive OOF encodings. Validation and test rows are encoded only
    from the model-training fold to avoid leaking validation labels.
    """
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    train_fold_cat = train_cat.loc[train_indices]
    y_train_fold = y.loc[train_indices]
    y_train_onehot = pd.get_dummies(y_train_fold, prefix="target")
    class_cols = y_train_onehot.columns.tolist()
    encoded_train = pd.DataFrame(index=train_cat.index)
    encoded_test = pd.DataFrame(index=test_cat.index)

    for col in train_cat.columns:
        for cls in class_cols:
            encoded_train[f"{col}__{cls}_te"] = 0.0
            encoded_test[f"{col}__{cls}_te"] = 0.0

        for tr_pos, val_pos in cv.split(train_fold_cat, y_train_fold):
            fold_train_idx = train_fold_cat.iloc[tr_pos].index
            fold_valid_idx = train_fold_cat.iloc[val_pos].index
            fold_stats = y_train_onehot.loc[fold_train_idx].groupby(train_cat.loc[fold_train_idx, col]).mean()
            fold_prior = y_train_onehot.loc[fold_train_idx].mean()
            mapped = train_cat.loc[fold_valid_idx, col].map(fold_stats.to_dict("index"))

            for cls in class_cols:
                encoded_train.loc[fold_valid_idx, f"{col}__{cls}_te"] = [
                    value.get(cls, fold_prior[cls]) if isinstance(value, dict) else fold_prior[cls]
                    for value in mapped
                ]

        full_stats = y_train_onehot.groupby(train_fold_cat[col]).mean()
        full_prior = y_train_onehot.mean()

        for target_index, target_cat, target_encoded in [
            (valid_indices, train_cat.loc[valid_indices], encoded_train),
            (test_cat.index, test_cat, encoded_test),
        ]:
            mapped = target_cat[col].map(full_stats.to_dict("index"))
            for cls in class_cols:
                target_encoded.loc[target_index, f"{col}__{cls}_te"] = [
                    value.get(cls, full_prior[cls]) if isinstance(value, dict) else full_prior[cls]
                    for value in mapped
                ]

    return encoded_train, encoded_test


X_raw_fe = add_domain_features(X_all_raw)
X_test_raw_fe = add_domain_features(test_clean)

num_cols = X_raw_fe.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X_raw_fe.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"Numeric columns after domain features: {len(num_cols)}")
print(f"Categorical columns after interactions: {len(cat_cols)}")

poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = pd.DataFrame(
    poly.fit_transform(X_raw_fe[num_cols]),
    columns=poly.get_feature_names_out(num_cols),
    index=X_raw_fe.index,
)
X_test_poly = pd.DataFrame(
    poly.transform(X_test_raw_fe[num_cols]),
    columns=poly.get_feature_names_out(num_cols),
    index=X_test_raw_fe.index,
)

X_cat_cv_encoded, X_test_cat_encoded = multiclass_target_encode(
    X_raw_fe[cat_cols],
    X_test_raw_fe[cat_cols],
    y_all,
    train_idx,
    valid_idx,
    n_splits=5,
)

# Shared engineered matrix for LightGBM / sklearn models.
X_engineered = pd.concat([X_poly, X_cat_cv_encoded], axis=1)
X_test_engineered = pd.concat([X_test_poly, X_test_cat_encoded], axis=1)

# Raw categorical matrix for CatBoost, preserving categorical columns.
X_catboost_raw = X_raw_fe.copy()
X_test_catboost = X_test_raw_fe.copy()

X_lgb_train, X_lgb_valid = X_engineered.iloc[train_idx], X_engineered.iloc[valid_idx]
X_cat_train, X_cat_valid = X_catboost_raw.iloc[train_idx], X_catboost_raw.iloc[valid_idx]
y_train, y_valid = y_all.iloc[train_idx], y_all.iloc[valid_idx]

print("\nFeature matrices:")
print(f"  CatBoost raw categorical: {X_catboost_raw.shape}")
print(f"  LightGBM engineered poly + target encoded: {X_engineered.shape}")
print(f"  LightGBM test engineered: {X_test_engineered.shape}")
print(f"\nTrain/valid split: {X_lgb_train.shape[0]:,} / {X_lgb_valid.shape[0]:,}")

Numeric columns after domain features: 24
Categorical columns after interactions: 13

Feature matrices:
  CatBoost raw categorical: (630000, 37)
  LightGBM engineered poly + target encoded: (630000, 363)
  LightGBM test engineered: (270000, 363)

Train/valid split: 504,000 / 126,000


In [5]:
# ============================================================================
# PART 3: TUNE CATBOOST + LIGHTGBM WITH GRIDSEARCH PIPELINES
# ============================================================================

CLASS_ORDER = sorted(y_all.unique())
TUNE_SAMPLE_N = 50_000


def stratified_sample_index(y, sample_n=TUNE_SAMPLE_N):
    sample_n = min(sample_n, len(y))
    return y.groupby(y, group_keys=False).apply(
        lambda s: s.sample(max(1, int(round(len(s) * sample_n / len(y)))), random_state=RANDOM_STATE)
    ).index


def align_proba(model, proba):
    if hasattr(model, "classes_"):
        model_classes = list(model.classes_)
    else:
        model_classes = list(model.named_steps["model"].classes_)

    aligned = np.zeros((proba.shape[0], len(CLASS_ORDER)))
    for i, cls in enumerate(CLASS_ORDER):
        aligned[:, i] = proba[:, model_classes.index(cls)]
    return aligned


def run_grid_search(name, pipeline, param_grid, X, y, fit_kwargs=None):
    fit_kwargs = fit_kwargs or {}
    sample_idx = stratified_sample_index(y)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring=PRIMARY_METRIC,
        cv=cv,
        n_jobs=1,
        verbose=1,
    )

    print(f"\n{name}: GridSearchCV on {len(sample_idx):,} sampled rows")
    grid.fit(X.loc[sample_idx], y.loc[sample_idx], **fit_kwargs)
    print(f"Best {name} CV balanced accuracy: {grid.best_score_:.5f}")
    print(f"Best {name} parameters: {grid.best_params_}")
    return grid


# CatBoost pipeline: no preprocessing step, because CatBoost handles categorical columns directly.
catboost_pipeline = Pipeline([
    ("model", CatBoostClassifier(
        loss_function="MultiClass",
        random_seed=RANDOM_STATE,
        verbose=0,
        thread_count=-1,
    )),
])
# Candidate sets centered on the best Optuna trials from catboost_boosting.ipynb.
# GridSearchCV will test whether those settings still win with the extra features.
catboost_grid = [
    {
        "model__iterations": [300],
        "model__learning_rate": [0.19871763652839242],
        "model__depth": [9],
        "model__l2_leaf_reg": [9.368123295483429],
        "model__bagging_temperature": [2.0366063374980534],
        "model__random_strength": [2.305201465490145],
        "model__border_count": [255],
    },
    {
        "model__iterations": [280],
        "model__learning_rate": [0.1871600494311255],
        "model__depth": [8],
        "model__l2_leaf_reg": [3.797136700063681],
        "model__bagging_temperature": [0.7892635862028128],
        "model__random_strength": [3.8643444057715977],
        "model__border_count": [207],
    },
    {
        "model__iterations": [299],
        "model__learning_rate": [0.19558481759380753],
        "model__depth": [8],
        "model__l2_leaf_reg": [5.960352890679389],
        "model__bagging_temperature": [1.8525994696332224],
        "model__random_strength": [3.6220264219694616],
        "model__border_count": [216],
    },
    {
        "model__iterations": [267],
        "model__learning_rate": [0.061056046995676974],
        "model__depth": [9],
        "model__l2_leaf_reg": [4.978102710304074],
        "model__bagging_temperature": [3.3733406253825593],
        "model__random_strength": [1.5792219439091746],
        "model__border_count": [254],
    },
]

# LightGBM pipeline: engineered features are already built, so the pipeline just wraps the model.
lgb_pipeline = Pipeline([
    ("model", lgb.LGBMClassifier(
        objective="multiclass",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
    )),
])
# Candidate sets centered on the best Optuna trials from lightGBM_boosting.ipynb.
lgb_grid = [
    {
        "model__n_estimators": [1162],
        "model__learning_rate": [0.018322389859678854],
        "model__num_leaves": [148],
        "model__max_depth": [8],
        "model__min_child_samples": [94],
        "model__subsample": [0.6771731571319778],
        "model__subsample_freq": [3],
        "model__colsample_bytree": [0.8866746919819607],
        "model__reg_alpha": [0.011112539657265613],
        "model__reg_lambda": [7.468507554972082],
    },
    {
        "model__n_estimators": [1124],
        "model__learning_rate": [0.02427171505816292],
        "model__num_leaves": [83],
        "model__max_depth": [10],
        "model__min_child_samples": [84],
        "model__subsample": [0.7036329435415349],
        "model__subsample_freq": [5],
        "model__colsample_bytree": [0.6577390086917289],
        "model__reg_alpha": [0.008375429546886509],
        "model__reg_lambda": [2.939909350972704],
    },
    {
        "model__n_estimators": [1372],
        "model__learning_rate": [0.19729052043986137],
        "model__num_leaves": [243],
        "model__max_depth": [13],
        "model__min_child_samples": [49],
        "model__subsample": [0.9386742619852532],
        "model__subsample_freq": [2],
        "model__colsample_bytree": [0.7943117371839126],
        "model__reg_alpha": [4.337300848961289],
        "model__reg_lambda": [0.00015374701700336915],
    },
    {
        "model__n_estimators": [919],
        "model__learning_rate": [0.07447222693552025],
        "model__num_leaves": [206],
        "model__max_depth": [8],
        "model__min_child_samples": [90],
        "model__subsample": [0.9626431289889903],
        "model__subsample_freq": [7],
        "model__colsample_bytree": [0.9536002471698768],
        "model__reg_alpha": [9.899855206770184e-08],
        "model__reg_lambda": [3.0267162934042375e-06],
    },
]

start = time.time()
catboost_search = run_grid_search(
    "CatBoost",
    catboost_pipeline,
    catboost_grid,
    X_cat_train,
    y_train,
    fit_kwargs={"model__cat_features": cat_cols},
)
catboost_model = clone(catboost_search.best_estimator_)
catboost_model.fit(X_cat_train, y_train, model__cat_features=cat_cols)
catboost_pred = np.asarray(catboost_model.predict(X_cat_valid)).ravel()
catboost_proba = align_proba(catboost_model, catboost_model.predict_proba(X_cat_valid))
catboost_time = time.time() - start

start = time.time()
lgb_search = run_grid_search(
    "LightGBM",
    lgb_pipeline,
    lgb_grid,
    X_lgb_train,
    y_train,
)
lgb_model = clone(lgb_search.best_estimator_)
lgb_model.fit(X_lgb_train, y_train)
lgb_pred = np.asarray(lgb_model.predict(X_lgb_valid)).ravel()
lgb_proba = align_proba(lgb_model, lgb_model.predict_proba(X_lgb_valid))
lgb_time = time.time() - start

model_results = pd.DataFrame([
    {
        "Model": "CatBoost",
        "Feature_Representation": "raw categorical features + domain features",
        "CV_BalancedAccuracy": catboost_search.best_score_,
        "Valid_BalancedAccuracy": balanced_accuracy_score(y_valid, catboost_pred),
        "Valid_Accuracy": accuracy_score(y_valid, catboost_pred),
        "Best_Params": catboost_search.best_params_,
        "Runtime_Minutes": catboost_time / 60,
    },
    {
        "Model": "LightGBM",
        "Feature_Representation": "polynomial numeric + target-encoded categorical features",
        "CV_BalancedAccuracy": lgb_search.best_score_,
        "Valid_BalancedAccuracy": balanced_accuracy_score(y_valid, lgb_pred),
        "Valid_Accuracy": accuracy_score(y_valid, lgb_pred),
        "Best_Params": lgb_search.best_params_,
        "Runtime_Minutes": lgb_time / 60,
    },
]).sort_values("Valid_BalancedAccuracy", ascending=False).reset_index(drop=True)

fitted_models = {
    "CatBoost": {
        "model": catboost_model,
        "X_train": X_cat_train,
        "X_valid": X_cat_valid,
        "X_test": X_test_catboost,
        "valid_pred": catboost_pred,
        "valid_proba": catboost_proba,
        "best_params": catboost_search.best_params_,
    },
    "LightGBM": {
        "model": lgb_model,
        "X_train": X_lgb_train,
        "X_valid": X_lgb_valid,
        "X_test": X_test_engineered,
        "valid_pred": lgb_pred,
        "valid_proba": lgb_proba,
        "best_params": lgb_search.best_params_,
    },
}

best_single_name = model_results.loc[0, "Model"]
best_single_ba = model_results.loc[0, "Valid_BalancedAccuracy"]

print("\n" + "=" * 70)
print("MODEL COMPARISON")
print("=" * 70)
display(model_results[["Model", "CV_BalancedAccuracy", "Valid_BalancedAccuracy", "Valid_Accuracy", "Runtime_Minutes"]])
print(f"Best individual model: {best_single_name} ({best_single_ba:.5f} balanced accuracy)")


CatBoost: GridSearchCV on 50,000 sampled rows
Fitting 3 folds for each of 4 candidates, totalling 12 fits
Best CatBoost CV balanced accuracy: 0.96025
Best CatBoost parameters: {'model__bagging_temperature': 0.7892635862028128, 'model__border_count': 207, 'model__depth': 8, 'model__iterations': 280, 'model__l2_leaf_reg': 3.797136700063681, 'model__learning_rate': 0.1871600494311255, 'model__random_strength': 3.8643444057715977}

LightGBM: GridSearchCV on 50,000 sampled rows
Fitting 3 folds for each of 4 candidates, totalling 12 fits
Best LightGBM CV balanced accuracy: 0.95654
Best LightGBM parameters: {'model__colsample_bytree': 0.9536002471698768, 'model__learning_rate': 0.07447222693552025, 'model__max_depth': 8, 'model__min_child_samples': 90, 'model__n_estimators': 919, 'model__num_leaves': 206, 'model__reg_alpha': 9.899855206770184e-08, 'model__reg_lambda': 3.0267162934042375e-06, 'model__subsample': 0.9626431289889903, 'model__subsample_freq': 7}

MODEL COMPARISON


,Model,CV_BalancedAccuracy,Valid_BalancedAccuracy,Valid_Accuracy,Runtime_Minutes
0,CatBoost,0.960247,0.964091,0.985135,21.703445
1,LightGBM,0.956543,0.963755,0.985817,21.716103


Best individual model: CatBoost (0.96409 balanced accuracy)


In [10]:
# ============================================================================
# PART 4: FEATURE USEFULNESS ANALYSIS
# ============================================================================

print("=" * 70)
print("FEATURE USEFULNESS ANALYSIS")
print("=" * 70)

importance_tables = {}

if "LightGBM" in fitted_models:
    lgb_pipeline_fit = fitted_models["LightGBM"]["model"]
    lgb_step = lgb_pipeline_fit.named_steps["model"]
    lgb_importance = pd.DataFrame({
        "Feature": X_engineered.columns,
        "LightGBM_Gain": lgb_step.booster_.feature_importance(importance_type="gain"),
    }).sort_values("LightGBM_Gain", ascending=False)
    lgb_importance["Rank"] = range(1, len(lgb_importance) + 1)
    importance_tables["LightGBM_Gain"] = lgb_importance

    print("\nTop LightGBM engineered features by gain:")
    display(lgb_importance.head(20))

if "CatBoost" in fitted_models:
    cat_pipeline_fit = fitted_models["CatBoost"]["model"]
    cat_step = cat_pipeline_fit.named_steps["model"]
    cat_importance = pd.DataFrame({
        "Feature": X_catboost_raw.columns,
        "CatBoost_Importance": cat_step.get_feature_importance(),
    }).sort_values("CatBoost_Importance", ascending=False)
    cat_importance["Rank"] = range(1, len(cat_importance) + 1)
    importance_tables["CatBoost"] = cat_importance

    print("\nTop CatBoost raw/categorical features:")
    display(cat_importance.head(20))

# Permutation importance is slower, so use a stratified validation sample.
perm_model_name = best_single_name
perm_model = fitted_models[perm_model_name]["model"]
perm_X_valid = fitted_models[perm_model_name]["X_valid"]
perm_sample_idx = stratified_sample_index(y_valid, sample_n=15_000)

print(f"\nPermutation importance for {perm_model_name} on {len(perm_sample_idx):,} validation rows")
perm = permutation_importance(
    perm_model,
    perm_X_valid.loc[perm_sample_idx],
    y_valid.loc[perm_sample_idx],
    n_repeats=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    scoring="balanced_accuracy",
)

permutation_df = pd.DataFrame({
    "Feature": perm_X_valid.columns,
    "Permutation_Mean": perm.importances_mean,
    "Permutation_Std": perm.importances_std,
}).sort_values("Permutation_Mean", ascending=False)
importance_tables["Permutation"] = permutation_df

print("\nTop permutation-importance features:")
display(permutation_df.head(20))

useful_features = permutation_df.loc[permutation_df["Permutation_Mean"] > 0, "Feature"].tolist()
print(f"\nPositive permutation features: {len(useful_features)} of {len(permutation_df)}")
print("\nInterpretation:")
print("- Target-encoded categorical variables show whether crop stage, mulching, soil/crop groups, or region carry class signal.")
print("- Weather and water variables such as moisture, temperature, rainfall, wind, and previous irrigation are expected drivers of irrigation need.")
print("- Polynomial and interaction terms are useful when they rank near the top; otherwise they still document tested feature representations.")

FEATURE USEFULNESS ANALYSIS

Top LightGBM engineered features by gain:


,Feature,LightGBM_Gain,Rank
330,Crop_Growth_Stage__target_High_te,1.369275e+06,1
1,Soil_Moisture,8.973835e+05,2
342,Mulching_Used__target_High_te,5.984022e+05,3
4,Temperature_C,4.508481e+05,4
8,Wind_Speed_kmh,3.151628e+05,5
203,Wind_Speed_kmh low_soil_moisture,2.831437e+05,6
198,Wind_Speed_kmh temp_sq,2.602196e+05,7
133,Temperature_C low_soil_moisture,1.971806e+05,8
170,Rainfall_mm low_soil_moisture,1.119583e+05,9
290,log_Rainfall_mm temp_sq,1.058153e+05,10



Top CatBoost raw/categorical features:


,Feature,CatBoost_Importance,Rank
11,Crop_Growth_Stage,27.840217,1
2,Soil_Moisture,23.655482,2
16,Mulching_Used,12.718036,3
9,Wind_Speed_kmh,9.932749,4
5,Temperature_C,7.198053,5
28,temp_sq,3.333666,6
24,Crop_Stage,1.828363,7
6,Humidity,1.439092,8
36,low_soil_moisture,1.056846,9
7,Rainfall_mm,1.049603,10



Permutation importance for CatBoost on 15,000 validation rows

Top permutation-importance features:


,Feature,Permutation_Mean,Permutation_Std
2,Soil_Moisture,0.347624,0.007324
11,Crop_Growth_Stage,0.294349,0.002168
16,Mulching_Used,0.181366,0.000918
9,Wind_Speed_kmh,0.169939,0.007776
5,Temperature_C,0.114761,0.002604
28,temp_sq,0.024081,0.001050
7,Rainfall_mm,0.016452,0.000807
19,Water_Available,0.014506,0.000274
20,Moisture_Deficit,0.004235,0.000474
26,log_Rainfall_mm,0.002513,0.000032



Positive permutation features: 30 of 37

Interpretation:
- Target-encoded categorical variables show whether crop stage, mulching, soil/crop groups, or region carry class signal.
- Weather and water variables such as moisture, temperature, rainfall, wind, and previous irrigation are expected drivers of irrigation need.
- Polynomial and interaction terms are useful when they rank near the top; otherwise they still document tested feature representations.


In [11]:
# ============================================================================
# PART 5: STACK CATBOOST AND LIGHTGBM
# ============================================================================

print("=" * 70)
print("CATBOOST + LIGHTGBM STACKING")
print("=" * 70)


def labels_from_proba(proba):
    return np.array(CLASS_ORDER)[np.argmax(proba, axis=1)]


# Simple probability averaging baseline.
avg_proba_valid = (catboost_proba + lgb_proba) / 2
avg_pred_valid = labels_from_proba(avg_proba_valid)

# Hold out part of the training fold for the meta-model.
stack_base_idx, stack_meta_idx = train_test_split(
    train_idx,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_all.iloc[train_idx],
)

catboost_stack_model = clone(catboost_pipeline).set_params(**catboost_search.best_params_)
catboost_stack_model.fit(
    X_catboost_raw.loc[stack_base_idx],
    y_all.loc[stack_base_idx],
    model__cat_features=cat_cols,
)

lgb_stack_model = clone(lgb_pipeline).set_params(**lgb_search.best_params_)
lgb_stack_model.fit(X_engineered.loc[stack_base_idx], y_all.loc[stack_base_idx])

cat_meta_proba = align_proba(
    catboost_stack_model,
    catboost_stack_model.predict_proba(X_catboost_raw.loc[stack_meta_idx]),
)
lgb_meta_proba = align_proba(
    lgb_stack_model,
    lgb_stack_model.predict_proba(X_engineered.loc[stack_meta_idx]),
)

X_stack_train = np.hstack([cat_meta_proba, lgb_meta_proba])
y_stack_train = y_all.loc[stack_meta_idx]

meta_model = LogisticRegression(
    max_iter=400,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)
meta_model.fit(X_stack_train, y_stack_train)

cat_valid_stack_proba = align_proba(
    catboost_stack_model,
    catboost_stack_model.predict_proba(X_cat_valid),
)
lgb_valid_stack_proba = align_proba(
    lgb_stack_model,
    lgb_stack_model.predict_proba(X_lgb_valid),
)
X_stack_valid = np.hstack([cat_valid_stack_proba, lgb_valid_stack_proba])
stack_pred_valid = meta_model.predict(X_stack_valid)

ensemble_results = pd.DataFrame([
    {
        "Model": "CatBoost",
        "Type": "single",
        "Valid_BalancedAccuracy": balanced_accuracy_score(y_valid, catboost_pred),
        "Valid_Accuracy": accuracy_score(y_valid, catboost_pred),
    },
    {
        "Model": "LightGBM",
        "Type": "single",
        "Valid_BalancedAccuracy": balanced_accuracy_score(y_valid, lgb_pred),
        "Valid_Accuracy": accuracy_score(y_valid, lgb_pred),
    },
    {
        "Model": "Average(CatBoost, LightGBM)",
        "Type": "average",
        "Valid_BalancedAccuracy": balanced_accuracy_score(y_valid, avg_pred_valid),
        "Valid_Accuracy": accuracy_score(y_valid, avg_pred_valid),
    },
    {
        "Model": "Stacking(CatBoost, LightGBM)",
        "Type": "stacking",
        "Valid_BalancedAccuracy": balanced_accuracy_score(y_valid, stack_pred_valid),
        "Valid_Accuracy": accuracy_score(y_valid, stack_pred_valid),
    },
]).sort_values("Valid_BalancedAccuracy", ascending=False).reset_index(drop=True)

best_overall_name = ensemble_results.loc[0, "Model"]
best_overall_type = ensemble_results.loc[0, "Type"]
best_single_ba = model_results["Valid_BalancedAccuracy"].max()
ensemble_results["Improvement_vs_BestSingle"] = ensemble_results["Valid_BalancedAccuracy"] - best_single_ba

display(ensemble_results)
print(f"Best validation approach: {best_overall_name}")

CATBOOST + LIGHTGBM STACKING


,Model,Type,Valid_BalancedAccuracy,Valid_Accuracy,Improvement_vs_BestSingle
0,"Stacking(CatBoost, LightGBM)",stacking,0.967714,0.984302,0.003623
1,CatBoost,single,0.964091,0.985135,0.000000
2,"Average(CatBoost, LightGBM)",average,0.963870,0.985849,-0.000221
3,LightGBM,single,0.963755,0.985817,-0.000336


Best validation approach: Stacking(CatBoost, LightGBM)


In [12]:
# ============================================================================
# PART 6: TEST PREDICTIONS AND KAGGLE SUBMISSION
# ============================================================================

print("=" * 70)
print("GENERATING TEST PREDICTIONS")
print("=" * 70)

cat_test_proba = align_proba(catboost_model, catboost_model.predict_proba(X_test_catboost))
lgb_test_proba = align_proba(lgb_model, lgb_model.predict_proba(X_test_engineered))

if best_overall_type == "single" and best_overall_name == "CatBoost":
    final_pred = np.asarray(catboost_model.predict(X_test_catboost)).ravel()
    final_strategy = "CatBoost"
elif best_overall_type == "single" and best_overall_name == "LightGBM":
    final_pred = np.asarray(lgb_model.predict(X_test_engineered)).ravel()
    final_strategy = "LightGBM"
elif best_overall_type == "average":
    final_pred = labels_from_proba((cat_test_proba + lgb_test_proba) / 2)
    final_strategy = "average CatBoost + LightGBM probabilities"
else:
    cat_stack_test_proba = align_proba(
        catboost_stack_model,
        catboost_stack_model.predict_proba(X_test_catboost),
    )
    lgb_stack_test_proba = align_proba(
        lgb_stack_model,
        lgb_stack_model.predict_proba(X_test_engineered),
    )
    X_stack_test = np.hstack([cat_stack_test_proba, lgb_stack_test_proba])
    final_pred = meta_model.predict(X_stack_test)
    final_strategy = "stacking CatBoost + LightGBM"

submission = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    TARGET_COL: final_pred,
})
submission.to_csv("submission_homework4_final.csv", index=False)

best_single_submission = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    TARGET_COL: np.asarray(
        fitted_models[best_single_name]["model"].predict(fitted_models[best_single_name]["X_test"])
    ).ravel(),
})
best_single_submission.to_csv("submission_homework4_best_single.csv", index=False)

print(f"Final submission strategy: {final_strategy}")
print(f"Prediction distribution: {submission[TARGET_COL].value_counts().to_dict()}")
print("\nCreated files:")
print("- submission_homework4_final.csv")
print(f"- submission_homework4_best_single.csv ({best_single_name})")
display(submission.head())

GENERATING TEST PREDICTIONS
Final submission strategy: stacking CatBoost + LightGBM
Prediction distribution: {'Low': 159581, 'Medium': 101366, 'High': 9053}

Created files:
- submission_homework4_final.csv
- submission_homework4_best_single.csv (CatBoost)


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


In [13]:
# ============================================================================
# PART 7: ASSIGNMENT SUMMARY
# ============================================================================

from IPython.display import Markdown

model_lines = []
for _, row in model_results.iterrows():
    model_lines.append(
        f"- **{row['Model']}**: {row['Feature_Representation']} "
        f"(valid balanced accuracy = {row['Valid_BalancedAccuracy']:.5f})"
    )

perm_top = importance_tables["Permutation"].head(8)["Feature"].tolist()
lgb_top = importance_tables.get("LightGBM_Gain", pd.DataFrame()).head(5).get("Feature", pd.Series(dtype=str)).tolist()
cat_top = importance_tables.get("CatBoost", pd.DataFrame()).head(5).get("Feature", pd.Series(dtype=str)).tolist()

best_ensemble = ensemble_results.iloc[0]
best_single_row = ensemble_results[ensemble_results["Type"] == "single"].iloc[0]
ensemble_delta = best_ensemble["Valid_BalancedAccuracy"] - best_single_row["Valid_BalancedAccuracy"]

summary_md = f"""
## Final Assignment Summary

### 1. Multiple Models Built and Tuned
{chr(10).join(model_lines)}

The two models differ in both algorithm and feature representation. CatBoost uses the raw categorical variables directly, while LightGBM uses polynomial numeric features plus target-encoded categorical variables. Both models are tuned with `GridSearchCV` pipelines using candidate parameter sets centered on the best Optuna trials from the CatBoost and LightGBM reference notebooks.

### 2. New Features Created and Tested
The notebook tests polynomial numeric interactions, leakage-safe multiclass target encodings, domain features such as `Water_Available`, `Moisture_Deficit`, `Heat_Dryness`, irrigation-per-hectare ratios, and categorical interactions such as `Crop_Stage` and `Soil_Crop`. It also adds Homework3-style transformations including log rainfall/irrigation features, squared temperature, humidity-temperature interactions, quantile bins, and binary high/low indicators.

### 3. Useful Feature Evidence
The strongest permutation-importance features were: {', '.join(perm_top)}. LightGBM's top gain features included: {', '.join(lgb_top)}. CatBoost's top raw/categorical features included: {', '.join(cat_top)}. Together these results support which original and engineered features contributed most to prediction performance.

### 4. Ensemble Result
Best individual validation balanced accuracy: **{best_single_row['Valid_BalancedAccuracy']:.5f}**. Best two-model strategy: **{best_ensemble['Model']}** with validation balanced accuracy **{best_ensemble['Valid_BalancedAccuracy']:.5f}**. Difference versus best single model: **{ensemble_delta:+.5f}**.

The final Kaggle file `submission_homework4_final.csv` uses the best validation strategy selected above, and `submission_homework4_best_single.csv` is saved as a comparison submission.
"""

Markdown(summary_md)


## Final Assignment Summary

### 1. Multiple Models Built and Tuned
- **CatBoost**: raw categorical features + domain features (valid balanced accuracy = 0.96409)
- **LightGBM**: polynomial numeric + target-encoded categorical features (valid balanced accuracy = 0.96375)

The two models differ in both algorithm and feature representation. CatBoost uses the raw categorical variables directly, while LightGBM uses polynomial numeric features plus target-encoded categorical variables. Both models are tuned with `GridSearchCV` pipelines using candidate parameter sets centered on the best Optuna trials from the CatBoost and LightGBM reference notebooks.

### 2. New Features Created and Tested
The notebook tests polynomial numeric interactions, leakage-safe multiclass target encodings, domain features such as `Water_Available`, `Moisture_Deficit`, `Heat_Dryness`, irrigation-per-hectare ratios, and categorical interactions such as `Crop_Stage` and `Soil_Crop`. It also adds Homework3-style transformations including log rainfall/irrigation features, squared temperature, humidity-temperature interactions, quantile bins, and binary high/low indicators.

### 3. Useful Feature Evidence
The strongest permutation-importance features were: Soil_Moisture, Crop_Growth_Stage, Mulching_Used, Wind_Speed_kmh, Temperature_C, temp_sq, Rainfall_mm, Water_Available. LightGBM's top gain features included: Crop_Growth_Stage__target_High_te, Soil_Moisture, Mulching_Used__target_High_te, Temperature_C, Wind_Speed_kmh. CatBoost's top raw/categorical features included: Crop_Growth_Stage, Soil_Moisture, Mulching_Used, Wind_Speed_kmh, Temperature_C. Together these results support which original and engineered features contributed most to prediction performance.

### 4. Ensemble Result
Best individual validation balanced accuracy: **0.96409**. Best two-model strategy: **Stacking(CatBoost, LightGBM)** with validation balanced accuracy **0.96771**. Difference versus best single model: **+0.00362**.

The final Kaggle file `submission_homework4_final.csv` uses the best validation strategy selected above, and `submission_homework4_best_single.csv` is saved as a comparison submission.
